# Training and Test Errors, Cross-Validation, Pipelines

In [1]:
import pandas as pd
import numpy as np

from plotnine import *

**Use scikit-learn, including pipelines, whenever possible.**

## Ames Housing Data


We have fit different regression models to predict house prices (SalePrice, not log). Now we will evaluate models by estimating their prediction error.

In [2]:
df_ames = pd.read_csv("http://dlsun.github.io/pods/data/AmesHousing.txt", sep = "\t")

df_ames

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


Since all of the houses in the data set have a SalePrice, we'll include all the observations in the training data. We'll start with a basic linear regression model with a few features.

In [3]:
X_train = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath"]]

y_train = df_ames["SalePrice"]

In [4]:
from sklearn.linear_model import LinearRegression

model1 = LinearRegression()
model1.fit(
    X=X_train,
    y=y_train
)

LinearRegression()

1. Explain conceptually what the "fit" step does. What is the output?

The fit step uses the training data to learn the relationship between the features and the outcome. The output is a trained model we can use to make predictions.

2. Explain conceptually how you would compute the *training* RMSE (root mean square error) of this model.

Predict the outcomes for the training data, square the differences between the actual and predicted values, average them, and take the square root.

3. Interpret the *training RMSE*. Is it big or small? What could you use as a rough benchmark for comparison?

Training RMSE shows roughly how far off the predictions are, in the outcome's units. We'd need its value to judge it, but predicting the mean for everyone gives us a simple benchmark.

4. Suppose we add more features like "TotRms AbvGrd" and fit a new model. Would the *training* RMSE increase or decrease? Why is comparing the *training* RMSE of these two models not useful?

For ordinary linear regression, training RMSE will decrease or stay the same when we add features. Comparing training errors isn't that useful because a model can fit the training data better without predicting new data better.

5. If we fit a 1-nearest neighbor model, what should the *training* RMSE be?

The training RMSE should be 0 because each observation is its own closest neighbor, so it predicts its own outcome exactly, assuming no conflicting duplicate feature values.

6. What happens to the *training* RMSE of a $k$-nearest neighbors model as $k$ increases?

Training RMSE generally increases as k increases because predictions average more neighbors and follow individual training points less closely.

7. We are really interested in the *test* RMSE, not the *training* RMSE. Explain why in practice you can not compute the *test* RMSE.

We can't know the exact error across all future observations because we haven't seen their actual outcomes yet. We can estimate it using held-out data or cross-validation.

8. Explain conceptually how cross-validations *estimates* the *test* RMSE.

Cross-validation repeatedly trains the model on some of the data and predicts the portion left out. Those prediction errors give us an estimate of how well it handles new data.

9. We'll use cross-validation to estimate the *test* RMSE for the model from part 1. Will the *test* RMSE be greater than, less than, or equal to the *training* RMSE for this model?

The estimated test RMSE will usually be higher than the training RMSE because the model hasn't already seen the observations it's being evaluated on.

10. Suppose we add features like we did in part 4. Will the *test* RMSE necessarily decrease? Fit the test RMSE and compare the two models - which is better based on this criteria?

Adding features won't necessarily lower test RMSE because extra features can cause overfitting. We'd need the cross-validation results for both models to choose the one with the lower estimated test RMSE.

11. Now suppose we add a categorical predictor like Neighborhood. The crossval code below will (probably) error. Explain why. Hint: think about how the cross-validation process works and how categorical variables are treated.

A neighborhood might appear in a validation fold without appearing in that fold's training data. That can cause an error because the model hasn't learned how to handle that category.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

ct = ColumnTransformer(
    [("standardize", StandardScaler(), ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath"],),
     ("dummify", OneHotEncoder(), ["Neighborhood"]),
    ],
    remainder="drop",
)

neighborhood_model = Pipeline(
    [("preprocessing", ct),
     ("lm", LinearRegression())]
)

X_train = df_ames[["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Neighborhood"]]
y_train = df_ames["SalePrice"]

neighborhood_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('standardize',
                                                  StandardScaler(),
                                                  ['Gr Liv Area',
                                                   'Bedroom AbvGr', 'Full Bath',
                                                   'Half Bath']),
                                                 ('dummify', OneHotEncoder(),
                                                  ['Neighborhood'])])),
                ('lm', LinearRegression())])

In [6]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(neighborhood_model,
                         X = X_train,
                         y = y_train,
                         scoring="neg_mean_squared_error",
                         cv=10)
scores

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 140, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 380, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~~~~~~~~~^
        estimator, *args, response_method=response_method, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/

array([-1.44545836e+09, -2.17595026e+09, -1.03958997e+09, -1.75337710e+09,
       -1.09461197e+09, -2.46144943e+09, -1.41980035e+09, -2.32075641e+09,
       -1.21633497e+09,             nan])

11. Going back to part 1, suppose we had standardized the features first. Would the coeffiecients of the fitted model change? Would the *training* RMSE change?

Yes, the coefficients would change because the features are on a different scale. For ordinary linear regression, the predictions stay the same, so the training RMSE wouldn't change.

12. Suppose we want to standardize predictors like in the previous part and estimate test RMSE using cross-validation. Explain how standardization fits in in the CV process. Also explain why it is important to put standardization as part of the model pipeline.

In each cross-validation round, calculate the mean and standard deviation using only the training fold, then use those values to standardize both the training and validation folds. Putting this in a pipeline handles it automatically and keeps information from the validation data from leaking into training.

## Practice Activity

We'll continue with the housing data and our goal of predicting Sale Price.

We will have a competition to see who can come up with the model with the smallest *test* RMSE. Fit several models and try to find the smallest *test* RMSE. Use cross-validation to estimate the *test* RMSE. When you think you have a good model, share your it and its test RMSE with a partner. Then see what other people have done; can you do even better?

Note: next week we will see how to make some of these choices in a more automated way. For now just work by trial and error. You should try:

- Exploring the data first! Make some plots and summaries to get an idea of what might be good predictors to try
- Several different sets of predictors, including both categorical and numerical predictors
- Linear regression models, including models with polynomial features and interactions
- K-nearest neighbors models, inclduing different values of k
- Decision trees, including different sizes of trees

Use pipelines as much as possible, and don't forget to transform variables as needed.

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import pandas as pd

numerical_cols = ["Gr Liv Area", "Lot Area", "Overall Qual", "Year Built", "Bedroom AbvGr", "Full Bath", "Half Bath"]
categorical_cols = ["Neighborhood", "Bldg Type", "House Style"]

X = df_ames[numerical_cols + categorical_cols]
y = df_ames["SalePrice"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

models = {
    "Linear Regression": Pipeline([
        ("pre", preprocessor),
        ("model", LinearRegression())
    ]),
    "Linear Regression (Poly/Interaction)": Pipeline([
        ("pre", preprocessor),
        ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "KNN (k=5)": Pipeline([
        ("pre", preprocessor),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]),
    "KNN (k=15)": Pipeline([
        ("pre", preprocessor),
        ("model", KNeighborsRegressor(n_neighbors=15))
    ]),
    "Decision Tree (Max Depth=5)": Pipeline([
        ("pre", preprocessor),
        ("model", DecisionTreeRegressor(max_depth=5, random_state=42))
    ]),
    "Decision Tree (Max Depth=10)": Pipeline([
        ("pre", preprocessor),
        ("model", DecisionTreeRegressor(max_depth=10, random_state=42))
    ])
}

results = {}
for name, pipeline in models.items():
    scores = cross_val_score(
        pipeline, X, y,
        scoring="neg_mean_squared_error",
        cv=10,
        n_jobs=-1
    )
    rmse_scores = np.sqrt(-scores)
    results[name] = rmse_scores.mean()
    print(f"{name}: Mean Test RMSE = {rmse_scores.mean():,.2f}")


Linear Regression: Mean Test RMSE = 32,483.68
Linear Regression (Poly/Interaction): Mean Test RMSE = 33,636.55
KNN (k=5): Mean Test RMSE = 32,610.17
KNN (k=15): Mean Test RMSE = 32,239.24
Decision Tree (Max Depth=5): Mean Test RMSE = 37,854.44
Decision Tree (Max Depth=10): Mean Test RMSE = 36,504.72
